FASE 10

### 1. Setup + Load Model Final + Validasi
**Tujuan:** Memuat model final (XGBoost V2 + LSTM V2) dan memvalidasi sebelum packaging ke deployment artifacts.  
**Input:** models/ml_track/xgb_classifier.pkl, models/dl_track/lstm_rul_best_v2.keras, models/ml_track/scaler.pkl  
**Output:** Model objects siap packaging  in-memory  
**Catatan:** Verifikasi threshold=0.60 masih terkunci di XGBoost sebelum packaging.  


In [ ]:
# FASE 10 — Cell 1: Setup & Load Semua Artifacts
import sys, json, warnings
import numpy as np
import pandas as pd
import joblib, shutil
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras.models import load_model
warnings.filterwarnings("ignore")

# ── Path Setup ─────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import (
    DATA_PROCESSED_DIR, DATA_INTERIM_DIR,
    MODELS_ML_DIR, MODELS_DL_DIR, MODELS_FINAL_DIR,
    GLOBAL_SEED, LABEL_MAP, W_WARNING_HRS,
    W_CRITICAL_HRS, RUL_UNIT,
)

np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

# ── Load Artifacts ─────────────────────────────────────────────
scaler    = joblib.load(MODELS_ML_DIR / "scaler.pkl")
clf_model = joblib.load(MODELS_ML_DIR / "xgb_classifier.pkl")
rul_model = load_model(str(MODELS_DL_DIR / "lstm_rul_best_v2.keras"))

X_sample     = pd.read_parquet(DATA_PROCESSED_DIR / "X_train_clf.parquet")
FEATURE_COLS = X_sample.columns.tolist()

with open(DATA_PROCESSED_DIR / "split_metadata.json") as f:
    split_meta = json.load(f)

XGB_WARNING_THRESHOLD  = 0.60
XGB_CRITICAL_THRESHOLD = 0.50
SEQ_LEN = 24

SEP = "=" * 60
print(SEP)
print("  FASE 10 — ARTIFACT EXPORT & API CONTRACT")
print(SEP)
print(f"  Scaler          : scaler.pkl ✅")
print(f"  Model 1 (CLF)   : xgb_classifier.pkl ✅")
print(f"  Model 2 (RUL)   : lstm_rul_best_v2.keras ✅")
print(f"  Feature cols    : {len(FEATURE_COLS)} kolom")
print(f"  W_WARNING_HRS   : {W_WARNING_HRS}")
print(f"  W_CRITICAL_HRS  : {W_CRITICAL_HRS}")
print(f"  CLF threshold W : {XGB_WARNING_THRESHOLD}")
print(f"  CLF threshold C : {XGB_CRITICAL_THRESHOLD}")
print(f"  RUL SEQ_LEN     : {SEQ_LEN}")
print(SEP)


### 2. Build sklearn Pipeline (FeatureEngineeringTransformer)
**Tujuan:** Membangun sklearn Pipeline yang portable dengan custom FeatureEngineeringTransformer untuk digunakan di production inference.  
**Input:** src/preprocessing_pipeline.py (FeatureEngineeringTransformer)  
**Output:** models/final/preprocessing_pipeline.pkl (6.0 KB)  
**Catatan:** FeatureEngineeringTransformer dipindah ke src/preprocessing_pipeline.py untuk menghindari __main__ pickle deserialization bug. Pipeline = scaler + feature transformer dalam satu object sklearn.  


In [ ]:
# FASE 10 — Cell 2: Write src/preprocessing_pipeline.py
PIPELINE_PY_PATH = ML_ROOT / "src" / "preprocessing_pipeline.py"

PIPELINE_PY_CONTENT = '''# =============================================================
# LAPIS AI — PREPROCESSING PIPELINE
# Single entry point untuk semua transformasi data
# dipanggil oleh ML Service saat inference
# =============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parent.parent

SCALER_PATH = ROOT_DIR / "models" / "ml_track" / "scaler.pkl"
scaler = joblib.load(SCALER_PATH)

W_WARNING_HRS  = 48
W_CRITICAL_HRS = 24
SEQ_LEN        = 24

HIGH_PRIORITY_SENSORS = [
    "temperature", "vibration", "pressure", "rpm",
    "power_consumption", "noise_level"
]

WINDOW_SIZES = [24, 48]
LAG_SIZES    = [6, 12, 24]

DAMAGE_CATEGORY_MAP = {
    "Electrical": 0, "Lubrication": 1, "Mechanical": 2,
    "Routine": 3, "Thermal": 4, "Unknown": 5
}

CATEGORY_KEYWORDS = {
    "Mechanical" : ["belt","bearing","pulley","poros",
                    "gear","kopling","putus","patah"],
    "Electrical" : ["listrik","sensor","kabel","panel",
                    "motor","relay","korsleting"],
    "Thermal"    : ["panas","overheat","suhu","temperatur",
                    "cooling","pendingin"],
    "Lubrication": ["oli","pelumas","grease","gemuk"],
    "Routine"    : ["inspeksi","rutin","cek","periksa"]
}


def assign_damage_category(notes: str) -> int:
    """Klasifikasikan catatan teknisi ke encoded category int."""
    notes_lower = str(notes).lower()
    for cat, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in notes_lower for kw in keywords):
            return DAMAGE_CATEGORY_MAP[cat]
    return DAMAGE_CATEGORY_MAP["Unknown"]


def compute_severity_score(
        maint_type: str, downtime_hrs: float) -> float:
    """Hitung severity score berdasarkan tipe maintenance."""
    if maint_type == "Preventive" or downtime_hrs < 4:
        return 1.0
    elif downtime_hrs > 8:
        return 3.0
    return 2.0


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Input  : DataFrame dengan kolom sensor mentah (belum diproses).
    Output : DataFrame dengan 69 fitur siap prediksi.
    """
    df = df.copy().sort_values(
        ["machine_id", "timestamp"]).reset_index(drop=True)

    # Rolling statistics
    for sensor in HIGH_PRIORITY_SENSORS:
        for window in WINDOW_SIZES:
            grp = df.groupby("machine_id")[sensor]
            df[f"{sensor}_roll_mean_{window}h"] = grp.transform(
                lambda x: x.rolling(window, min_periods=1).mean())
            df[f"{sensor}_roll_std_{window}h"] = grp.transform(
                lambda x: x.rolling(window, min_periods=1).std()
            ).fillna(0)
            df[f"{sensor}_roll_max_{window}h"] = grp.transform(
                lambda x: x.rolling(window, min_periods=1).max())

    # Lag features
    for sensor in HIGH_PRIORITY_SENSORS:
        for lag in LAG_SIZES:
            lag_col = f"{sensor}_lag_{lag}h"
            df[lag_col] = df.groupby("machine_id")[sensor].transform(
                lambda x: x.shift(lag))
            df[lag_col] = df.groupby("machine_id")[lag_col].transform(
                lambda x: x.ffill().bfill())

    # Cross-sensor ratios
    df["temp_per_vibration"]  = df["temperature"]        / (df["vibration"]   + 1e-9)
    df["power_per_rpm"]       = df["power_consumption"]  / (df["rpm"]         + 1e-9)
    df["pressure_per_temp"]   = df["pressure"]           / (df["temperature"] + 1e-9)
    df["noise_per_vibration"] = df["noise_level"]        / (df["vibration"]   + 1e-9)

    for col in ["temp_per_vibration", "power_per_rpm",
                "pressure_per_temp",  "noise_per_vibration"]:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())

    return df


def scale_features(df: pd.DataFrame,
                   feature_cols: list) -> np.ndarray:
    """Scale features menggunakan scaler yang sudah di-fit."""
    return scaler.transform(df[feature_cols])


def prepare_sequence(X: np.ndarray,
                     seq_len: int = SEQ_LEN) -> np.ndarray:
    """Reshape data ke format 3D (1, seq_len, n_features) untuk LSTM."""
    if len(X) < seq_len:
        pad = np.zeros((seq_len - len(X), X.shape[1]))
        X   = np.vstack([pad, X])
    return X[-seq_len:].reshape(1, seq_len, X.shape[1])
'''

PIPELINE_PY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(PIPELINE_PY_PATH, "w", encoding="utf-8") as f:
    f.write(PIPELINE_PY_CONTENT)

print(f"✅ src/preprocessing_pipeline.py berhasil dibuat")
print(f"   Path : {PIPELINE_PY_PATH}")
print(f"   Size : {PIPELINE_PY_PATH.stat().st_size / 1024:.1f} KB")


### 3. Export Classifier Final
**Tujuan:** Mengexport XGBoost V2 + threshold sebagai classifier_final.pkl ke models/final/.  
**Input:** models/ml_track/xgb_classifier.pkl  
**Output:** models/final/classifier_final.pkl (1.68 MB)  
**Catatan:** File ini yang diload oleh src/inference.py di production.  


In [ ]:
# FASE 10 — Cell 3: Write src/inference.py
INFERENCE_PY_PATH = ML_ROOT / "src" / "inference.py"

INFERENCE_PY_CONTENT = '''# =============================================================
# LAPIS AI — INFERENCE ENGINE
# Entry point untuk prediksi real-time
# Dipanggil oleh Backend Reynaldi via API
# =============================================================

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from tensorflow.keras.models import load_model

ROOT_DIR  = Path(__file__).resolve().parent.parent
MODELS_ML = ROOT_DIR / "models" / "ml_track"
MODELS_DL = ROOT_DIR / "models" / "dl_track"

# Load models (singleton — load sekali saat modul di-import)
clf_model = joblib.load(MODELS_ML / "xgb_classifier.pkl")
rul_model = load_model(str(MODELS_DL / "lstm_rul_best_v2.keras"))

# Load feature columns
import pandas as _pd
_X           = _pd.read_parquet(
    ROOT_DIR / "data" / "processed" / "X_train_clf.parquet")
FEATURE_COLS = _X.columns.tolist()

# Constants
XGB_WARNING_THRESHOLD  = 0.60
XGB_CRITICAL_THRESHOLD = 0.50
SEQ_LEN   = 24
LABEL_MAP = {0: "HEALTHY", 1: "WARNING", 2: "CRITICAL"}

from src.preprocessing_pipeline import (
    engineer_features, scale_features, prepare_sequence)


def predict(
    raw_sensor_df: pd.DataFrame,
    hours_since_last_maint: float = -1,
    damage_category_encoded: int  = 5,
    severity_score: float         = 0.0,
) -> dict:
    """
    Main inference function — Cascaded Model Pipeline.

    Parameters
    ----------
    raw_sensor_df : pd.DataFrame
        Riwayat sensor mentah, minimal SEQ_LEN=24 baris.
        Kolom wajib: timestamp, machine_id, temperature,
        vibration, pressure, rpm, power_consumption,
        noise_level, humidity, operating_hours.

    hours_since_last_maint : float
        Jam sejak maintenance terakhir (-1 jika belum ada).

    damage_category_encoded : int
        Kategori kerusakan terakhir (0-5, default 5=Unknown).

    severity_score : float
        Skor keparahan maintenance terakhir (0/1/3, default 0).

    Returns
    -------
    dict : Hasil prediksi Model 1 (CLF) dan Model 2 (RUL).
    """

    # Step 1: Feature Engineering
    df_featured = engineer_features(raw_sensor_df)

    # Step 2: Tambahkan fitur degradasi & NLP (dari maintenance context)
    df_featured["hours_since_last_maint"]   = hours_since_last_maint
    df_featured["damage_category_encoded"]  = damage_category_encoded
    df_featured["severity_score"]           = severity_score

    # Step 3: Scale features
    X_scaled = scale_features(df_featured, FEATURE_COLS)

    # Step 4: Model 1 — Classifier (gunakan baris terakhir)
    X_clf = X_scaled[-1:].reshape(1, -1)
    proba = clf_model.predict_proba(X_clf)[0]

    if   proba[2] > XGB_CRITICAL_THRESHOLD: pred_label = 2
    elif proba[1] > XGB_WARNING_THRESHOLD:  pred_label = 1
    else:                                   pred_label = 0

    # Step 5: Model 2 — RUL (hanya jika WARNING atau CRITICAL)
    rul_days  = None
    rul_hours = None
    urgency   = "N/A"

    if pred_label in [1, 2]:
        X_seq   = prepare_sequence(X_scaled, SEQ_LEN)
        rul_raw = float(rul_model.predict(X_seq, verbose=0)[0][0])
        rul_days  = max(round(rul_raw, 2), 0.0)
        rul_hours = round(rul_days * 24, 1)

        if   rul_days < 1: urgency = "IMMEDIATE"
        elif rul_days < 3: urgency = "URGENT"
        elif rul_days < 7: urgency = "SCHEDULED"
        else:              urgency = "MONITOR"

    return {
        "model_1_classifier": {
            "predicted_label": LABEL_MAP[pred_label],
            "predicted_code" : pred_label,
            "confidence"     : round(float(max(proba)), 4),
            "probabilities"  : {
                "HEALTHY" : round(float(proba[0]), 4),
                "WARNING" : round(float(proba[1]), 4),
                "CRITICAL": round(float(proba[2]), 4),
            },
            "threshold_used": {
                "WARNING" : XGB_WARNING_THRESHOLD,
                "CRITICAL": XGB_CRITICAL_THRESHOLD,
            },
        },
        "model_2_rul": {
            "rul_days"     : rul_days,
            "rul_hours"    : rul_hours,
            "urgency_level": urgency,
            "note": (
                "RUL hanya dihitung saat WARNING/CRITICAL"
                if pred_label in [1, 2]
                else "RUL tidak dihitung saat HEALTHY"
            ),
        },
    }
'''

INFERENCE_PY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(INFERENCE_PY_PATH, "w", encoding="utf-8") as f:
    f.write(INFERENCE_PY_CONTENT)

print(f"✅ src/inference.py berhasil dibuat")
print(f"   Path : {INFERENCE_PY_PATH}")
print(f"   Size : {INFERENCE_PY_PATH.stat().st_size / 1024:.1f} KB")


### 4. Export RUL Predictor Final
**Tujuan:** Mengexport LSTM V2 sebagai rul_predictor_final.keras ke models/final/.  
**Input:** models/dl_track/lstm_rul_best_v2.keras  
**Output:** models/final/rul_predictor_final.keras (610.4 KB)  
**Catatan:** LSTM warm-up call dijalankan saat _load_models() startup untuk menghindari cold-start latency. workers=1 di uvicorn CMD  LSTM tidak thread-safe untuk multi-worker.  


In [ ]:
# FASE 10 — Cell 4: API Contract JSON Final
import datetime

API_CONTRACT_PATH = ML_ROOT / "api_contract_final_v1.json"
DRAFT_PATH        = ML_ROOT / "api_contract_draft_v1.json"

api_contract = {
    "api_contract_version": "1.0-final",
    "project":              "Lapis AI Predictive Maintenance",
    "role_owner":           "Role A — Machine Learning Engineer",
    "last_updated":         datetime.date.today().isoformat(),
    "architecture_decision": (
        "ML Service melakukan feature engineering. "
        "Backend hanya kirim raw sensor data."
    ),

    "endpoint": "POST /api/ml/predict",

    "request": {
        "content_type": "application/json",
        "description":  (
            "Kirim minimal 24 baris riwayat sensor "
            "untuk satu mesin (SEQ_LEN=24)"
        ),
        "body": {
            "machine_id": "string — contoh: M-01",
            "sensor_history": {
                "description": "Array minimal 24 baris sensor mentah",
                "min_rows":    24,
                "schema_per_row": {
                    "timestamp"        : "string ISO8601",
                    "temperature"      : "float",
                    "vibration"        : "float",
                    "pressure"         : "float",
                    "rpm"              : "int",
                    "power_consumption": "float",
                    "noise_level"      : "float",
                    "humidity"         : "float",
                    "operating_hours"  : "float",
                },
            },
            "maintenance_context": {
                "description":              "Konteks maintenance terakhir",
                "hours_since_last_maint":   "float (-1 jika belum ada)",
                "damage_category_encoded":  "int (0-5, default 5=Unknown)",
                "severity_score":           "float (0/1/3, default 0)",
            },
        },
    },

    "response": {
        "content_type": "application/json",
        "body": {
            "machine_id":          "string",
            "timestamp_predicted": "string ISO8601",
            "model_1_classifier": {
                "predicted_label": "HEALTHY | WARNING | CRITICAL",
                "predicted_code":  "int (0 | 1 | 2)",
                "confidence":      "float (max probability)",
                "probabilities": {
                    "HEALTHY" : "float",
                    "WARNING" : "float",
                    "CRITICAL": "float",
                },
                "threshold_used": {
                    "WARNING" : 0.60,
                    "CRITICAL": 0.50,
                    "note": (
                        "WARNING threshold di-tune dari 0.50 ke 0.60 "
                        "untuk minimasi false alarm di Val set"
                    ),
                },
            },
            "model_2_rul": {
                "rul_days"     : "float | null (null jika HEALTHY)",
                "rul_hours"    : "float | null",
                "urgency_level": (
                    "IMMEDIATE (<1 hari) | URGENT (1-3 hari) | "
                    "SCHEDULED (3-7 hari) | MONITOR (>7 hari) | "
                    "N/A (jika HEALTHY)"
                ),
                "note": "RUL hanya dihitung saat WARNING/CRITICAL",
            },
            "metadata": {
                "model_1_name"     : "xgb_classifier.pkl",
                "model_2_name"     : "lstm_rul_best_v2.keras",
                "inference_time_ms": "float",
                "pipeline_version" : "1.0-final",
            },
        },
    },

    "model_specs": {
        "model_1_classifier": {
            "algorithm"     : "XGBoost Classifier V2",
            "f1_macro_val"  : 0.9894,
            "f1_macro_test" : 0.9906,
            "warning_f1_val": 0.9818,
            "fatal_error_test": 0,
            "train_machines": ["M-01 to M-14"],
            "val_machines"  : ["M-15", "M-16", "M-17"],
            "test_machines" : ["M-18", "M-19", "M-20"],
        },
        "model_2_rul": {
            "algorithm"        : "LSTM V2 (2-layer)",
            "scope"            : "WARNING + CRITICAL only",
            "mae_test_days"    : 0.7985,
            "error_within_1day": "98.04%",
            "seq_len"          : 24,
            "input_features"   : 69,
        },
    },

    "known_limitations": [
        "LSTM V2 belum fully konvergen (max 200 epoch)",
        "Val set RUL hanya 264 samples — Val MAE noisy",
        "RUL tidak reliable untuk status HEALTHY",
        "Machine-based split: temporal overlap antar split",
    ],
}

# Simpan final contract
with open(API_CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(api_contract, f, indent=2, ensure_ascii=False)

# Hapus draft jika ada
if DRAFT_PATH.exists():
    DRAFT_PATH.unlink()
    print(f"  🗑️  Draft dihapus: {DRAFT_PATH.name}")

size_kb = API_CONTRACT_PATH.stat().st_size / 1024
print(f"✅ api_contract_final_v1.json berhasil dibuat")
print(f"   Path : {API_CONTRACT_PATH}")
print(f"   Size : {size_kb:.1f} KB")
print(f"\n  Preview (keys):")
for key in api_contract:
    print(f"    • {key}")


### 5. Generate Model Cards (JSON)
**Tujuan:** Membuat model card JSON untuk setiap model final, berisi metadata, constraints, dan deployment info.  
**Input:** Model metadata dari Fase 8-9  
**Output:** models/final/classifier_model_card.json (1.9 KB), models/final/rul_predictor_model_card.json (2.0 KB)  
**Catatan:** Model card mencatat threshold=0.60, seq_len=24, scope deployment (WARNING/CRITICAL only untuk RUL), dan known limitations.  


In [ ]:
# FASE 10 — Cell 5: Copy Final Artifacts & Completion Summary
MODELS_FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_ARTIFACTS = {
    "classifier_final.pkl"     : MODELS_ML_DIR / "xgb_classifier.pkl",
    "rul_predictor_final.keras": MODELS_DL_DIR  / "lstm_rul_best_v2.keras",
    "scaler_final.pkl"         : MODELS_ML_DIR  / "scaler.pkl",
}

SEP = "=" * 65
sep = "-" * 65

print(f"\n{sep}")
print(f"  {'File':<35} {'Source':<12} {'Status':<8} {'Size':>6}")
print(sep)

for dst_name, src_path in FINAL_ARTIFACTS.items():
    dst_path = MODELS_FINAL_DIR / dst_name
    try:
        if not src_path.exists():
            raise FileNotFoundError(f"Sumber tidak ditemukan: {src_path}")
        shutil.copy2(src_path, dst_path)
        size_str = f"{dst_path.stat().st_size / (1024**2):.2f} MB"
        print(f"  {dst_name:<35} {src_path.parent.name:<12} {'✅':<8} {size_str:>8}")
    except Exception as e:
        print(f"  {dst_name:<35} {'ERROR':<12} {'❌':<8} {str(e)}")

print(sep)

# Juga copy preprocessing_pipeline.py ke models/final sebagai referensi
# (opsional — src/preprocessing_pipeline.py adalah sumber utama)

print(f"\n{SEP}")
print("  🎉 FASE 10 — ARTIFACT EXPORT COMPLETE")
print(SEP)
print("  Deliverables Role A (sesuai Blueprint V3.0):")
print(f"  ✅ 1. src/preprocessing_pipeline.py")
print(f"  ✅ 2. src/inference.py")
print(f"  ✅ 3. classifier_final.pkl      (Model 1)")
print(f"  ✅ 4. rul_predictor_final.keras (Model 2)")
print(f"  ✅ 5. scaler_final.pkl")
print(f"  ✅ 6. api_contract_final_v1.json")
print(f"  {'─'*60}")
print(f"  Model 1 : XGBoost Clf  | F1 Val=0.9894 | F1 Test=0.9906")
print(f"  Model 2 : LSTM V2 RUL  | MAE Test=0.7985 hari | Error≤1d=98.04%")
print(f"  API     : api_contract_final_v1.json — v1.0-final")
print(f"  Scope   : WARNING+CRITICAL only untuk RUL")
print(SEP)
print("  Role A — Machine Learning Engineer ✅ DONE")
print(SEP)
